# Autenticación básica

Muchos servidores OPeNDAP requieren autenticación. [PyDAP](https://github.com/pydap/pydap) usa la biblioteca [requests](https://requests.readthedocs.io/en/latest/) de Python para obtener datos a través de la web, y también maneja dos formas generales de autenticación:


1. `username/password`.
2. `tokens`

### Requisito

- Un archivo `.netrc` en tu sistema local para guardar tus credenciales de autenticación (en máquinas Windows se llama `_netrc`).

```{note}
A continuación demostraremos cómo crear un archivo `.netrc` en tu sistema local.
```

Recomendamos firmemente tener un archivo local y estático `~/.netrc` que contenga todas las credenciales de autenticación, ya que [requests](https://requests.readthedocs.io/en/latest/) puede manejar por ti la autenticación desde un archivo `.netrc`, sin que tengas que hacer pasos adicionales. Es decir, `pydap` puede "descubrir" tus credenciales de autenticación OPeNDAP (usuario/contraseña) de forma eficiente si están definidas correctamente en un archivo `.netrc` (`_netrc` en máquinas Windows).

Aunque no todos los servidores OPeNDAP están configurados para autenticarse mediante `token`, la gran mayoría de los servidores Hyrax de NASA sí lo están, y el token se crea desde tu cuenta Earth Data Login (EDL). Entre los dos enfoques anteriores, la `autenticación con token` se recomienda generalmente a lo largo de la documentación porque puede evitar muchas redirecciones (aunque no todas).


```{note}
Para una introducción más amplia a la autenticación, consulta la documentación de `OPeNDAP` sobre [Authentication for DAP Clients](https://opendap.github.io/documentation/tutorials/ClientAuthentication.html) y [Client Authentication With EDLTokens](https://opendap.github.io/documentation/tutorials/ClientAuthenticationWithEDLTokens.html).
```


### ¡Accedamos a algunos datos remotos!


In [ ]:
from pydap.client import open_url
from pydap.net import create_session

In [ ]:
dataset_url = "https://opendap.earthdata.nasa.gov/providers/POCLOUD/collections/ECCO%20Ocean%20Temperature%20and%20Salinity%20-%20Monthly%20Mean%200.5%20Degree%20(Version%204%20Release%204)/granules/OCEAN_TEMPERATURE_SALINITY_mon_mean_2017-12_ECCO_V4r4_latlon_0p50deg"

## <font size="5"><span style='color:#0066cc'> **username / password**<font size="3">

No se necesita nada más que iniciar un objeto `requests.Session`. `pydap.net` tiene una función que puede iniciar una sesión de este tipo junto con algunos parámetros adicionales. Esto es:


In [ ]:
my_session = create_session() # default requests.session() object. 
print(my_session)

o si quieres cachear la sesión para evitar descargas repetidas:


In [ ]:
my_session = create_session(use_cache=True) # default requests_cache.session() object. 
print(my_session)

```{note}
tanto [requests](https://requests.readthedocs.io/en/latest/) como [requests_cache](https://requests-cache.readthedocs.io/en/stable/) pueden recuperar y manejar autenticación mediante `.netrc`
```


### Ahora abrimos el dataset remoto

Ahora, todo lo que se necesita es:

```python
ds = open_url(dataset_url, session=my_session, protocol="dap4")
```

La sesión de requests (predeterminada) recuperará las credenciales de .netrc y no se necesita información adicional.


## <font size="5"><span style='color:#0066cc'> **Tokens**<font size="3">

Este es otro enfoque soportado por varias instituciones y se usará a lo largo de la documentación. Para habilitar la autenticación con token, primero debes asegurarte de tener un token válido (no expirado) y, si no, crear uno nuevo. Por ejemplo, consulta este [recurso EDL](https://opendap.github.io/documentation/tutorials/ClientAuthenticationWithEDLTokens.html) específico de NASA.

Para mejorar la experiencia de uso, `pydap.net.create_session` puede recibir argumentos extra de sesión, como el `token`, para:

- Crear una `requests.session` con información del token en sus encabezados.
- Deshabilitar el descubrimiento automático del archivo `.netrc`.


In [ ]:
session_extra = {"token": "YourToken"}

# initialize a requests.session object with the token headers. All handled by pydap.
my_session = create_session(session_kwargs=session_extra)

```python
ds = open_url(dataset_url, session=my_session, protocol='dap4')
```


```{note}
En este punto solo se han descargado los metadatos para generar el dataset de pydap y se han agregado referencias a cada variable remota.
```


## <font size="5"><span style='color:#0066cc'> **earthaccess**<font size="3">

`earthaccess` permite ocultar la abstracción asociada con la recuperación de credenciales EDL. Pydap puede heredarlas desde el método de autenticación de earthaccess. Consulta el tutorial complementario.
